# ASTRAIOS RAG Pipeline — Build FAISS Index

This notebook is designed to run in **Google Colab**.  
It installs all dependencies, loads chunked documents, generates embeddings
using `sentence-transformers`, builds a FAISS index, and saves it to
Google Drive for persistent storage across sessions.

---

## 1. Install Dependencies

In [ ]:
!pip install -q llama-index-core llama-index-readers-file pypdf sentence-transformers faiss-cpu numpy

## 2. Mount Google Drive

We mount Google Drive so that:
- Raw documents can be read from a Drive folder
- The FAISS index can be saved persistently

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── Configure paths ──────────────────────────────────────────────
# Adjust these to match your Drive layout
DRIVE_BASE       = '/content/drive/MyDrive/ASTRAIOS'
RAW_DATA_DIR     = f'{DRIVE_BASE}/data/raw'
INDEX_SAVE_DIR   = f'{DRIVE_BASE}/faiss_index'

import os
os.makedirs(INDEX_SAVE_DIR, exist_ok=True)
print(f'Raw data dir : {RAW_DATA_DIR}')
print(f'Index save dir: {INDEX_SAVE_DIR}')

## 3. Configuration Constants

In [ ]:
# Must match ingestion/config.py
CHUNK_SIZE           = 512
CHUNK_OVERLAP        = 50
EMBEDDING_MODEL_NAME = 'all-MiniLM-L6-v2'
SUPPORTED_EXTENSIONS = ['.txt', '.pdf']

## 4. Load & Chunk Documents

In [ ]:
from pathlib import Path
from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import SentenceSplitter

data_dir = Path(RAW_DATA_DIR)
input_files = []
for ext in SUPPORTED_EXTENSIONS:
    input_files.extend(data_dir.glob(f'*{ext}'))

print(f'Found {len(input_files)} files:')
for f in input_files:
    print(f'  - {f.name}')

reader = SimpleDirectoryReader(input_files=[str(f) for f in input_files])
documents = reader.load_data()
print(f'\nLoaded {len(documents)} document(s)')

splitter = SentenceSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
nodes = splitter.get_nodes_from_documents(documents)
print(f'Split into {len(nodes)} chunks')

## 5. Generate Embeddings

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(EMBEDDING_MODEL_NAME)
print(f'Model loaded: {EMBEDDING_MODEL_NAME}')

texts = [node.get_content() for node in nodes]
embeddings = model.encode(texts, normalize_embeddings=True, show_progress_bar=True, batch_size=64)
embeddings = np.array(embeddings, dtype=np.float32)
print(f'Embeddings shape: {embeddings.shape}')

## 6. Build & Save FAISS Index

In [ ]:
import faiss
import pickle

dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)  # Inner product (cosine) on L2-normalized vectors
index.add(embeddings)
print(f'FAISS index: {index.ntotal} vectors, dim={dimension}')

# Save the FAISS index
faiss.write_index(index, f'{INDEX_SAVE_DIR}/index.faiss')
print(f'Index saved to {INDEX_SAVE_DIR}/index.faiss')

# Save chunk metadata alongside the index
chunk_meta = []
for i, node in enumerate(nodes):
    chunk_meta.append({
        'text': node.get_content(),
        'source_file': node.metadata.get('file_name', node.metadata.get('file_path', 'unknown')),
        'chunk_index': i,
    })

with open(f'{INDEX_SAVE_DIR}/chunks.pkl', 'wb') as f:
    pickle.dump(chunk_meta, f)

print(f'Chunk metadata saved ({len(chunk_meta)} entries)')
print('\n✅ Done! Index is saved to Google Drive.')

## 7. Quick Sanity Check

Run a quick similarity search to verify the index works.

In [ ]:
# Reload index from disk (to simulate what Retriever does)
saved_index = faiss.read_index(f'{INDEX_SAVE_DIR}/index.faiss')
with open(f'{INDEX_SAVE_DIR}/chunks.pkl', 'rb') as f:
    saved_chunks = pickle.load(f)

query = 'How old is the universe?'
q_emb = model.encode([query], normalize_embeddings=True).astype(np.float32)
distances, indices = saved_index.search(q_emb, 3)

print(f'Query: "{query}"\n')
for rank, (dist, idx) in enumerate(zip(distances[0], indices[0]), 1):
    chunk = saved_chunks[idx]
    snippet = chunk['text'][:150].replace('\n', ' ')
    print(f'  #{rank}  [score={dist:.4f}]  {chunk["source_file"]}')
    print(f'        {snippet}…\n')